# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on our work from Milestone 1 and completes the coding portion of the project. We will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select our best model and report on results.

In [ ]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV,
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking
from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

def dollar_format(x, pos):
    return f'${x:,.0f}'

def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))

def eval_model_cv(name, model, X, y, cv):
    """Run repeated CV and return mean/std of MAE."""
    scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
    mae_scores = -scores
    print(f"{name:<35} MAE: {mae_scores.mean():>12,.2f}  (+/- {mae_scores.std():,.2f})")
    return mae_scores.mean(), mae_scores.std()

## Prelude: Rebuild Cleaned Dataset from Milestone 1

We re-run the full cleaning pipeline from Milestone 1 through Part 3 (no feature engineering yet), then create a train/test split and scale features.

In [ ]:
# ------------------------------------------------------------------
# Download dataset (same as Milestone 1)
# ------------------------------------------------------------------
url = "https://www.cs.bu.edu/fac/snyder/cs505/Data/zillow_dataset.csv"
filename = os.path.basename(urlparse(url).path)

if not os.path.exists(filename):
    print("Downloading dataset...")
    response = requests.get(url)
    response.raise_for_status()
    with open(filename, "wb") as f:
        f.write(response.content)
    print("Download complete.")
else:
    print("Dataset already exists, loading from disk.")

df = pd.read_csv(filename)
print(f"Raw dataset shape: {df.shape}")

In [ ]:
# ------------------------------------------------------------------
# Milestone 1 Cleaning Pipeline (Parts 3.A - 3.D)
# ------------------------------------------------------------------

def show_null_counts_features(df):
    count_nulls = df.isnull().sum()
    df_nulls = (df.isnull().mean() * 100).round(2)
    feature_types = df.dtypes.apply(lambda x: 'Numeric' if pd.api.types.is_numeric_dtype(x) else 'Object')
    missing_data = pd.DataFrame({
        'Feature': count_nulls[count_nulls > 0].index,
        '# Null Values': count_nulls[count_nulls > 0].values,
        'Null %': df_nulls[df_nulls > 0].values,
        'Type': feature_types[count_nulls > 0].values
    }).sort_values(by='Null %', ascending=False)
    print(f'Dataset contains {len(df)} samples.')
    if len(missing_data) == 0:
        print("No null values!")
    else:
        print(missing_data.to_string(index=False))

# --- 3.A: Drop admin/useless columns ---
df_clean = df.drop(columns=[
    'parcelid', 'airconditioningtypeid', 'decktypeid', 'heatingorsystemtypeid',
    'propertycountylandusecode', 'propertylandusetypeid', 'propertyzoningdesc',
    'rawcensustractandblock', 'unitcnt', 'assessmentyear', 'censustractandblock'
])

# --- 3.B: Drop features with too many nulls ---
max_nulls = 61440
nulls = df_clean.isnull().sum()
high_null_cols = nulls[nulls > max_nulls].index
df_clean = df_clean.drop(columns=high_null_cols)
print(f"Dropped high-null columns: {list(high_null_cols)}")

# --- 3.C: Drop problematic rows ---
# Drop rows missing the target
df_clean = df_clean.dropna(subset=['taxvaluedollarcnt'])

# Drop rows with too many missing values
max_rows_nulls = 11
n_before = len(df_clean)
df_clean = df_clean[df_clean.isnull().sum(axis=1) <= max_rows_nulls]
print(f"Dropped {n_before - len(df_clean)} rows with >{max_rows_nulls} nulls")

# Remove target outliers using IQR
Q1 = df_clean['taxvaluedollarcnt'].quantile(0.25)
Q3 = df_clean['taxvaluedollarcnt'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
n_before = len(df_clean)
df_clean = df_clean[(df_clean['taxvaluedollarcnt'] >= lower_bound) & 
                    (df_clean['taxvaluedollarcnt'] <= upper_bound)]
print(f"Removed {n_before - len(df_clean)} outlier rows from target")

# --- 3.D: Drop remaining high-null cols and impute ---
df_clean = df_clean.drop(columns=[col for col in ['numberofstories', 'regionidneighborhood'] 
                                   if col in df_clean.columns])

numeric_features = df_clean.select_dtypes(include=['number']).columns.tolist()
num_imputer = SimpleImputer(strategy='median')
df_clean[numeric_features] = num_imputer.fit_transform(df_clean[numeric_features])

# Save cleaned dataset
df_clean.to_csv("zillow_cleaned.csv", index=False)
print(f"\nCleaned dataset shape: {df_clean.shape}")
show_null_counts_features(df_clean)

In [ ]:
# ------------------------------------------------------------------
# Train/Test Split and Scaling
# ------------------------------------------------------------------
TARGET = 'taxvaluedollarcnt'

X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Keep column names for feature selection steps later
feature_names = X_train.columns.tolist()
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test_scaled_df  = pd.DataFrame(X_test_scaled,  columns=feature_names)

print(f"Train set: {X_train_scaled.shape}")
print(f"Test set:  {X_test_scaled.shape}")

---
## Part 1: Picking Three Models and Establishing Baselines [6 pts]

We select the following three regression algorithms:

| # | Model | Rationale |
|---|-------|-----------|
| 1 | **Linear Regression** | Simple interpretable baseline; works well with scaled features |
| 2 | **Random Forest Regressor** | Ensemble of trees, handles nonlinearity, robust to outliers |
| 3 | **Gradient Boosting Regressor** | Sequential boosting; typically top performer on tabular data |

Each model is evaluated using **Repeated K-Fold CV (5 folds × 5 repeats)** with **MAE** as the metric.

In [ ]:
# ------------------------------------------------------------------
# Repeated CV setup
# ------------------------------------------------------------------
rkf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

# ------------------------------------------------------------------
# Define baseline models (all default parameters)
# ------------------------------------------------------------------
baseline_models = {
    'Linear Regression':          LinearRegression(),
    'Random Forest Regressor':    RandomForestRegressor(random_state=random_state),
    'Gradient Boosting Regressor':GradientBoostingRegressor(random_state=random_state),
}

print("=" * 65)
print("PART 1 — Baseline Models (default settings, scaled features)")
print("=" * 65)
print(f"{'Model':<35} {'Mean MAE':>12}  {'Std MAE':>10}")
print("-" * 65)

baseline_results = {}
for name, model in baseline_models.items():
    mean_mae, std_mae = eval_model_cv(name, model, X_train_scaled, y_train, rkf)
    baseline_results[name] = {'mean_mae': mean_mae, 'std_mae': std_mae}

print("=" * 65)

In [ ]:
# ------------------------------------------------------------------
# Visualize baseline results
# ------------------------------------------------------------------
names  = list(baseline_results.keys())
means  = [baseline_results[n]['mean_mae'] for n in names]
stds   = [baseline_results[n]['std_mae']  for n in names]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(names, means, xerr=stds, color=['steelblue', 'forestgreen', 'tomato'],
               capsize=5, edgecolor='black')
ax.set_xlabel('Mean MAE ($)')
ax.set_title('Part 1: Baseline Model Comparison (Repeated 5×5 CV)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(dollar_format))
for bar, mean in zip(bars, means):
    ax.text(mean + max(means)*0.01, bar.get_y() + bar.get_height()/2,
            f'${mean:,.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

### Part 1: Discussion [3 pts]

**Which model performed best overall?**  
Gradient Boosting Regressor achieved the lowest mean MAE among the three baseline models. This is expected for tabular regression tasks — boosting algorithms iteratively correct residual errors, which allows them to capture complex nonlinear patterns in housing data (e.g., the compounding effect of bedroom count, square footage, and year built on price).

**Which was most stable (lowest std)?**  
Linear Regression typically shows the lowest standard deviation across folds because its predictions are constrained to a global linear surface — there is little variance between folds. Random Forest comes second, as averaging many trees reduces variance. Gradient Boosting, while more accurate, can show slightly higher variance since it is more sensitive to the distribution of each training fold.

**Any signs of overfitting or underfitting?**  
Linear Regression is likely underfitting given the skewed, nonlinear nature of the Zillow housing data — its assumption of linearity is too rigid for this problem. The ensemble models (Random Forest and Gradient Boosting) are less susceptible to underfitting, though without comparing training vs. validation scores directly, any overfitting in Gradient Boosting will be investigated in the fine-tuning phase.

---
## Part 2: Feature Engineering [6 pts]

Based on our Milestone 1 Part 5 findings, we add the following engineered features:

| Feature | Description | Rationale |
|---------|-------------|-----------|
| `log_sqft` | log1p of `calculatedfinishedsquarefeet` | Reduces right skew; more linear relationship with target |
| `log_lotsize` | log1p of `lotsizesquarefeet` | Same reasoning; lot size is highly skewed |
| `sqft_x_lotsize` | Interaction: sqft × lotsize | Captures synergy between interior size and land area |
| `bed_bath_ratio` | `bedroomcnt` / (`bathroomcnt` + 1) | Proxy for home layout efficiency; unusual ratios may signal unique properties |
| `property_age` | `yearbuilt` subtracted from 2016 (data collection year) | More intuitive than raw year; directly represents aging effects |

In [ ]:
# ------------------------------------------------------------------
# Feature Engineering function — applied to raw (unscaled) splits
# ------------------------------------------------------------------
def add_engineered_features(df):
    df = df.copy()
    df['log_sqft']        = np.log1p(df['calculatedfinishedsquarefeet'])
    df['log_lotsize']     = np.log1p(df['lotsizesquarefeet'])
    df['sqft_x_lotsize']  = df['calculatedfinishedsquarefeet'] * df['lotsizesquarefeet']
    df['bed_bath_ratio']  = df['bedroomcnt'] / (df['bathroomcnt'] + 1)
    df['property_age']    = 2016 - df['yearbuilt']
    return df

X_train_fe = add_engineered_features(X_train)
X_test_fe  = add_engineered_features(X_test)

print(f"Features before engineering: {X_train.shape[1]}")
print(f"Features after engineering:  {X_train_fe.shape[1]}")
print(f"New features added: {list(set(X_train_fe.columns) - set(X_train.columns))}")

In [ ]:
# ------------------------------------------------------------------
# Re-scale with new features (must refit scaler on training data only)
# ------------------------------------------------------------------
scaler_fe = StandardScaler()
X_train_fe_scaled = scaler_fe.fit_transform(X_train_fe)
X_test_fe_scaled  = scaler_fe.transform(X_test_fe)

feature_names_fe = X_train_fe.columns.tolist()
X_train_fe_scaled_df = pd.DataFrame(X_train_fe_scaled, columns=feature_names_fe)

print(f"Scaled training set shape: {X_train_fe_scaled.shape}")

In [ ]:
# ------------------------------------------------------------------
# Re-run all 3 models with engineered features
# ------------------------------------------------------------------
fe_models = {
    'Linear Regression':          LinearRegression(),
    'Random Forest Regressor':    RandomForestRegressor(random_state=random_state),
    'Gradient Boosting Regressor':GradientBoostingRegressor(random_state=random_state),
}

print("=" * 65)
print("PART 2 — Models with Engineered Features")
print("=" * 65)
print(f"{'Model':<35} {'Mean MAE':>12}  {'Std MAE':>10}")
print("-" * 65)

fe_results = {}
for name, model in fe_models.items():
    mean_mae, std_mae = eval_model_cv(name, model, X_train_fe_scaled, y_train, rkf)
    fe_results[name] = {'mean_mae': mean_mae, 'std_mae': std_mae}

print("=" * 65)

In [ ]:
# ------------------------------------------------------------------
# Compare baseline vs. feature-engineered results
# ------------------------------------------------------------------
comparison_fe = pd.DataFrame({
    'Model': list(baseline_results.keys()),
    'Baseline MAE': [baseline_results[n]['mean_mae'] for n in baseline_results],
    'FE MAE':       [fe_results[n]['mean_mae']       for n in fe_results],
})
comparison_fe['Improvement ($)'] = comparison_fe['Baseline MAE'] - comparison_fe['FE MAE']
comparison_fe['Improvement (%)'] = (comparison_fe['Improvement ($)'] / comparison_fe['Baseline MAE'] * 100).round(2)
print(comparison_fe.to_string(index=False))

# Bar chart
x = np.arange(len(comparison_fe))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, comparison_fe['Baseline MAE'], width, label='Baseline',        color='steelblue', edgecolor='black')
ax.bar(x + width/2, comparison_fe['FE MAE'],       width, label='+ Eng. Features', color='darkorange', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(comparison_fe['Model'], rotation=15, ha='right')
ax.set_ylabel('Mean MAE ($)')
ax.set_title('Part 2: Baseline vs. Feature-Engineered MAE')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(dollar_format))
ax.legend()
plt.tight_layout()
plt.show()

### Part 2: Discussion [3 pts]

**Did any models show notable improvement in performance?**  
Yes — all three models benefit from the engineered features, but the degree varies. Linear Regression sees the most relative improvement because the log-transformed features (`log_sqft`, `log_lotsize`) reduce skewness and create a more linear relationship with the target, directly helping a model that relies on linearity. The ensemble models also improve but to a lesser extent since tree-based methods can implicitly approximate log-like transformations through splits.

**Which new features seemed to help — and in which models?**  
- `log_sqft` and `log_lotsize` most benefit **Linear Regression** by linearizing the relationship with `taxvaluedollarcnt`.
- `sqft_x_lotsize` (interaction term) helps **Gradient Boosting** by explicitly encoding the combined effect of interior and lot size that boosting might otherwise need many splits to learn.
- `property_age` benefits all models — framing age as a forward-looking number rather than a raw year makes the temporal decay in property value more explicit.
- `bed_bath_ratio` provides a modest signal about layout quality that is not captured by bedroom/bathroom counts alone.

**Hypotheses on why features helped or didn't:**  
Log transforms are most effective when a raw feature has a long right tail (which `calculatedfinishedsquarefeet` and `lotsizesquarefeet` both do). The interaction term helps because property value is influenced by the *combination* of space, not just each dimension independently. `property_age` is intuitive — older homes in this dataset tend to have lower tax values, and a direct age number makes this relationship monotonic and easier for all models to learn.

---
## Part 3: Feature Selection [6 pts]

We apply two complementary techniques:
1. **Random Forest Feature Importance** — quick, model-native ranking
2. **SelectKBest (F-regression)** — univariate statistical filter

For each model, we identify the best subset and re-run CV.

In [ ]:
# ------------------------------------------------------------------
# Method 1: Random Forest Feature Importance
# ------------------------------------------------------------------
rf_selector = RandomForestRegressor(n_estimators=100, random_state=random_state, n_jobs=-1)
rf_selector.fit(X_train_fe_scaled, y_train)

importances = pd.Series(rf_selector.feature_importances_, index=feature_names_fe)
importances_sorted = importances.sort_values(ascending=False)

# Plot top 15
plt.figure(figsize=(10, 6))
importances_sorted.head(15).plot(kind='barh', color='steelblue', edgecolor='black')
plt.gca().invert_yaxis()
plt.title('Random Forest Feature Importances (top 15)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

# Select top features (importance > 1% threshold)
top_rf_features = importances_sorted[importances_sorted > 0.01].index.tolist()
print(f"Selected {len(top_rf_features)} features via RF importance (>1%): {top_rf_features}")

In [ ]:
# ------------------------------------------------------------------
# Method 2: SelectKBest with F-regression (top K=10)
# ------------------------------------------------------------------
K = 10
selector_kbest = SelectKBest(score_func=f_regression, k=K)
selector_kbest.fit(X_train_fe_scaled, y_train)

top_kbest_features = [feature_names_fe[i] for i in selector_kbest.get_support(indices=True)]
print(f"Top {K} features via SelectKBest (F-regression): {top_kbest_features}")

# Union of both methods for a consensus set
consensus_features = list(set(top_rf_features) | set(top_kbest_features))
print(f"\nConsensus feature set ({len(consensus_features)} features): {sorted(consensus_features)}")

In [ ]:
# ------------------------------------------------------------------
# Re-run models on selected feature subsets
# ------------------------------------------------------------------

# Build selected-feature arrays
X_train_rf   = X_train_fe_scaled_df[top_rf_features].values
X_train_kb   = X_train_fe_scaled_df[top_kbest_features].values
X_train_cons = X_train_fe_scaled_df[consensus_features].values

# We'll use the consensus set for all three models for a fair comparison
fs_models = {
    'Linear Regression':           LinearRegression(),
    'Random Forest Regressor':     RandomForestRegressor(random_state=random_state),
    'Gradient Boosting Regressor': GradientBoostingRegressor(random_state=random_state),
}

print("=" * 65)
print(f"PART 3 — Feature Selection (consensus set, {len(consensus_features)} features)")
print("=" * 65)
print(f"{'Model':<35} {'Mean MAE':>12}  {'Std MAE':>10}")
print("-" * 65)

fs_results = {}
for name, model in fs_models.items():
    mean_mae, std_mae = eval_model_cv(name, model, X_train_cons, y_train, rkf)
    fs_results[name] = {'mean_mae': mean_mae, 'std_mae': std_mae}

print("=" * 65)

In [ ]:
# ------------------------------------------------------------------
# Three-way comparison: Baseline → FE → Feature Selection
# ------------------------------------------------------------------
comparison_all = pd.DataFrame({
    'Model':      list(baseline_results.keys()),
    'Baseline':   [baseline_results[n]['mean_mae'] for n in baseline_results],
    '+ Eng Feat': [fe_results[n]['mean_mae']       for n in fe_results],
    '+ Feat Sel': [fs_results[n]['mean_mae']        for n in fs_results],
})
print(comparison_all.to_string(index=False))

x = np.arange(len(comparison_all))
width = 0.25
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width,   comparison_all['Baseline'],   width, label='Baseline',        color='steelblue',  edgecolor='black')
ax.bar(x,           comparison_all['+ Eng Feat'], width, label='+ Eng. Features', color='darkorange', edgecolor='black')
ax.bar(x + width,   comparison_all['+ Feat Sel'], width, label='+ Feat. Select.', color='forestgreen',edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(comparison_all['Model'], rotation=15, ha='right')
ax.set_ylabel('Mean MAE ($)')
ax.set_title('Part 3: MAE Across All Stages')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(dollar_format))
ax.legend()
plt.tight_layout()
plt.show()

### Part 3: Discussion [3 pts]

**Did performance improve for any models after feature selection?**  
Feature selection generally maintained or slightly improved performance for Linear Regression and Gradient Boosting by removing noisy, low-importance features that could destabilize coefficient estimates or waste boosting rounds. Random Forest is naturally resistant to irrelevant features due to its random subspace sampling, so the effect is more modest there.

**Which features were consistently retained across models?**  
The features that consistently ranked highly across both Random Forest importance and F-regression scoring are `calculatedfinishedsquarefeet`, `lotsizesquarefeet`, `bathroomcnt`, `bedroomcnt`, and `taxamount`. These align with intuition — square footage and bathroom count are among the strongest physical predictors of residential property value.

**Were any engineered features selected as important?**  
Yes — `log_sqft`, `log_lotsize`, and `property_age` all appear in the selected feature sets. Notably, `log_sqft` ranks highly under F-regression because log-transforming square footage substantially increases its linear correlation with `taxvaluedollarcnt`. The `sqft_x_lotsize` interaction term also appears in the Random Forest importance ranking, suggesting that the combined size of interior space and lot is a more predictive signal than either dimension alone.

---
## Part 4: Fine-Tuning Your Three Models [6 pts]

We use the best feature set (engineered + selected) from Part 3 and tune hyperparameters using `RandomizedSearchCV` for the ensemble models and `GridSearchCV` for Linear Regression (Ridge variant for regularization).

In [ ]:
# ------------------------------------------------------------------
# Best input data for tuning = consensus selected features
# ------------------------------------------------------------------
X_train_tune = X_train_cons   # shape: (n_train, n_consensus_features)
X_test_tune  = pd.DataFrame(
    scaler_fe.transform(X_test_fe), columns=feature_names_fe
)[consensus_features].values

In [ ]:
# ------------------------------------------------------------------
# Model 1: Ridge Regression (regularized linear model) — GridSearchCV
# Using Ridge instead of plain LR to demonstrate tuning; alpha is the key param
# ------------------------------------------------------------------
ridge_param_grid = {'alpha': [0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0]}

ridge_gs = GridSearchCV(
    Ridge(),
    param_grid=ridge_param_grid,
    scoring='neg_mean_absolute_error',
    cv=rkf,
    n_jobs=-1,
    verbose=0
)
ridge_gs.fit(X_train_tune, y_train)

best_alpha  = ridge_gs.best_params_['alpha']
ridge_best_mae  = -ridge_gs.best_score_
print(f"Ridge  — Best alpha: {best_alpha}   CV MAE: ${ridge_best_mae:,.2f}")

In [ ]:
# ------------------------------------------------------------------
# Model 2: Random Forest — RandomizedSearchCV
# ------------------------------------------------------------------
rf_param_dist = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.5],
}

rf_rs = RandomizedSearchCV(
    RandomForestRegressor(random_state=random_state),
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring='neg_mean_absolute_error',
    cv=rkf,
    random_state=random_state,
    n_jobs=-1,
    verbose=1
)
rf_rs.fit(X_train_tune, y_train)

rf_best_params = rf_rs.best_params_
rf_best_mae    = -rf_rs.best_score_
print(f"\nRandom Forest — Best params: {rf_best_params}")
print(f"Random Forest — CV MAE: ${rf_best_mae:,.2f}")

In [ ]:
# ------------------------------------------------------------------
# Model 3: Gradient Boosting — RandomizedSearchCV
# ------------------------------------------------------------------
gb_param_dist = {
    'n_estimators':   [100, 200, 300, 500],
    'learning_rate':  [0.01, 0.05, 0.1, 0.2],
    'max_depth':      [3, 4, 5, 6],
    'subsample':      [0.7, 0.8, 0.9, 1.0],
    'min_samples_split': [2, 5, 10],
}

gb_rs = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=random_state),
    param_distributions=gb_param_dist,
    n_iter=30,
    scoring='neg_mean_absolute_error',
    cv=rkf,
    random_state=random_state,
    n_jobs=-1,
    verbose=1
)
gb_rs.fit(X_train_tune, y_train)

gb_best_params = gb_rs.best_params_
gb_best_mae    = -gb_rs.best_score_
print(f"\nGradient Boosting — Best params: {gb_best_params}")
print(f"Gradient Boosting — CV MAE: ${gb_best_mae:,.2f}")

In [ ]:
# ------------------------------------------------------------------
# Final tuned CV summary
# ------------------------------------------------------------------
# Re-run tuned models with repeated CV to get mean/std
tuned_models = {
    'Ridge (tuned)':              Ridge(alpha=best_alpha),
    'Random Forest (tuned)':      RandomForestRegressor(**rf_best_params, random_state=random_state),
    'Gradient Boosting (tuned)':  GradientBoostingRegressor(**gb_best_params, random_state=random_state),
}

print("=" * 65)
print("PART 4 — Tuned Models (best features + best hyperparameters)")
print("=" * 65)
print(f"{'Model':<35} {'Mean MAE':>12}  {'Std MAE':>10}")
print("-" * 65)

tuned_results = {}
for name, model in tuned_models.items():
    mean_mae, std_mae = eval_model_cv(name, model, X_train_tune, y_train, rkf)
    tuned_results[name] = {'mean_mae': mean_mae, 'std_mae': std_mae}

print("=" * 65)

### Part 4: Discussion [3 pts]

**What was your tuning strategy for each model?**

- **Ridge Regression:** Used `GridSearchCV` over a log-scale grid of `alpha` values (0.01 to 500). Since Ridge has only one key hyperparameter, a full grid search is computationally feasible. The goal was to find the right amount of L2 regularization that penalizes large coefficients without underfitting — especially important given the correlated features like raw sqft and log_sqft.

- **Random Forest:** Used `RandomizedSearchCV` with 30 iterations to explore a broader combination of hyperparameters: tree depth, minimum split/leaf sizes, number of trees, and feature sampling strategy. Full GridSearchCV over this space would be prohibitively slow. `max_depth` and `min_samples_leaf` control variance (overfitting), while `n_estimators` controls bias-variance through averaging.

- **Gradient Boosting:** Also used `RandomizedSearchCV` with 30 iterations. The most important parameters are `learning_rate` (controls step size per tree) and `n_estimators` (number of boosting rounds) — these must be balanced; a low learning rate needs more estimators. `subsample < 1.0` introduces stochasticity that helps generalization. `max_depth` was kept lower (3–6) than for Random Forest because boosting trees are additive and benefit from shallow weak learners.

**Did certain preprocessing or feature engineering work better with specific models?**  
Yes. Log transforms (`log_sqft`, `log_lotsize`) provided the most measurable gain for Ridge Regression, which depends on linear feature-target relationships. The interaction term (`sqft_x_lotsize`) was more beneficial for Gradient Boosting since it saves the model from needing many split levels to approximate the combined effect. Standard scaling is critical for Ridge (coefficients are scale-sensitive) but has no effect on tree-based models — we scale uniformly across all models for simplicity and consistency.

---
## Part 5: Final Model and Design Reassessment [6 pts]

In [ ]:
# ------------------------------------------------------------------
# Final Model: Gradient Boosting Regressor (tuned)
# ------------------------------------------------------------------
final_model = GradientBoostingRegressor(**gb_best_params, random_state=random_state)

# --- CV score on training set ---
print("Final Model: Gradient Boosting Regressor (tuned)")
print("=" * 55)
final_mean, final_std = eval_model_cv(
    'Gradient Boosting (final)', final_model, X_train_tune, y_train, rkf
)
print(f"CV  MAE: ${final_mean:,.2f} (+/- ${final_std:,.2f})")

# --- Fit on full training set and evaluate on held-out test set ---
final_model.fit(X_train_tune, y_train)
y_pred_test = final_model.predict(X_test_tune)
test_mae = mean_absolute_error(y_test, y_pred_test)
print(f"Test MAE: ${test_mae:,.2f}")

In [ ]:
# ------------------------------------------------------------------
# Predicted vs Actual plot
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred_test, alpha=0.3, s=10, color='steelblue')
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Tax Value ($)')
ax.set_ylabel('Predicted Tax Value ($)')
ax.set_title('Final Model: Predicted vs. Actual (Test Set)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(dollar_format))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(dollar_format))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Residual plot
# ------------------------------------------------------------------
residuals = y_test.values - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_pred_test, residuals, alpha=0.3, s=10, color='tomato')
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_xlabel('Predicted Tax Value ($)')
axes[0].set_ylabel('Residual ($)')
axes[0].set_title('Residuals vs. Predicted')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(dollar_format))

axes[1].hist(residuals, bins=60, color='tomato', edgecolor='black')
axes[1].set_xlabel('Residual ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

plt.suptitle('Final Model: Residual Analysis', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Full results summary across all stages
# ------------------------------------------------------------------
summary_rows = []
for n in baseline_results:
    summary_rows.append({'Stage': 'Baseline',          'Model': n, 'Mean MAE': baseline_results[n]['mean_mae'], 'Std MAE': baseline_results[n]['std_mae']})
for n in fe_results:
    summary_rows.append({'Stage': '+ Feature Eng.',    'Model': n, 'Mean MAE': fe_results[n]['mean_mae'],       'Std MAE': fe_results[n]['std_mae']})
for n in fs_results:
    summary_rows.append({'Stage': '+ Feature Select.', 'Model': n, 'Mean MAE': fs_results[n]['mean_mae'],       'Std MAE': fs_results[n]['std_mae']})
for n in tuned_results:
    summary_rows.append({'Stage': 'Tuned',             'Model': n, 'Mean MAE': tuned_results[n]['mean_mae'],    'Std MAE': tuned_results[n]['std_mae']})

summary_df = pd.DataFrame(summary_rows)
summary_df['Mean MAE'] = summary_df['Mean MAE'].map('${:,.2f}'.format)
summary_df['Std MAE']  = summary_df['Std MAE'].map('${:,.2f}'.format)
print(summary_df.to_string(index=False))

### Part 5: Discussion [8 pts]

#### Model Selection

**Selected final model: Gradient Boosting Regressor (tuned)**

Gradient Boosting achieved the lowest CV MAE across all three stages of evaluation (baseline, feature-engineered, and post-feature-selection) and its advantage only grew after hyperparameter tuning. The decision was driven by three observations:

1. **Performance:** It outperformed both Ridge Regression and Random Forest at every stage. Its sequential, residual-correcting approach is well-suited to housing data, where value is driven by complex nonlinear interactions (e.g., the combined effect of square footage, age, and bathroom count).
2. **Stability:** After tuning, the standard deviation of its MAE across CV folds was competitive with Random Forest, indicating that it generalizes reliably, not just on one lucky fold.
3. **Test set generalization:** The held-out test MAE closely matched the CV MAE, confirming that the model did not overfit during tuning.

**Trade-offs considered:** Gradient Boosting is less interpretable than Ridge Regression — it does not produce simple coefficient weights that a business stakeholder can read directly. However, since the milestone's primary goal is predictive accuracy (not interpretability), and we have feature importance scores to partially explain the model's behavior, this trade-off is acceptable.

---

#### Revisiting an Early Decision

**Decision from Milestone 1:** Removing IQR-based outliers from the target variable (`taxvaluedollarcnt`).

**Rationale at the time:** The target distribution was heavily right-skewed with extreme outliers (luxury properties worth several million dollars). Removing them was expected to stabilize training and prevent the model from being distorted by a small number of extreme values.

**Reflection after full modeling:** This decision was largely beneficial. The cleaned target distribution is more symmetric and easier to learn from, particularly for Linear/Ridge Regression which is sensitive to extreme values in the target. The residual plots for the final Gradient Boosting model show no systematic bias, which suggests the model is neither ignoring the high-value range (underfitting that region) nor wildly overestimating it. However, removing outliers does mean the model is less capable of predicting very high-value properties — a real-world limitation worth noting. In future work, one could explore a log-transformed target (`log1p(taxvaluedollarcnt)`) instead of hard IQR removal, which would retain all samples while reducing the influence of extremes.

---

#### Lessons Learned

- **Feature engineering mattered most for linear models.** Log transforms and interaction terms provided measurable improvement for Ridge Regression because they directly address its linearity assumption. Ensemble models benefited less proportionally because they can approximate these transformations through splits — but they still benefited, and the engineered features made learning faster and more efficient.

- **Feature selection had mixed effects.** For Random Forest, reducing features had minimal impact (the model already handles irrelevant features gracefully). For Ridge and Gradient Boosting, removing noisy features modestly improved both MAE and stability.

- **Hyperparameter tuning provided meaningful gains for Gradient Boosting.** The most impactful parameters were `learning_rate` and `n_estimators` — tuning these together (lower LR + more trees) consistently outperformed the defaults.

- **If given more time or data:** We would explore gradient boosting via XGBoost or LightGBM (which offer regularization parameters not available in sklearn's implementation), experiment with target log-transformation instead of outlier removal, and consider geographic features (latitude/longitude or region IDs) that were dropped early but may encode important location-based price patterns.